# Module 15: Feature Engineering

**Lesson: Scaling, Encoding, Imputation, Feature Creation, and Feature Selection**

This notebook covers essential feature engineering techniques for building high-performing ML models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, LabelEncoder, OrdinalEncoder,
    PolynomialFeatures, KBinsDiscretizer, FunctionTransformer
)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import (
    SelectKBest, f_classif, RFE, VarianceThreshold
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded successfully')

## 1. Numerical Feature Scaling

Compare StandardScaler, MinMaxScaler, and RobustScaler on California Housing data.

In [ ]:
housing = fetch_california_housing(as_frame=True)
X_h = housing.data[['MedInc', 'HouseAge', 'AveRooms', 'Population']]

scalers = {
    'Original': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, scaler) in zip(axes.flat, scalers.items()):
    if scaler is None:
        data = X_h
    else:
        data = pd.DataFrame(scaler.fit_transform(X_h), columns=X_h.columns)
    data.boxplot(ax=ax)
    ax.set_title(name)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print('StandardScaler: zero mean, unit variance. Best for normally distributed data.')
print('MinMaxScaler: scales to [0,1]. Best for bounded data or neural networks.')
print('RobustScaler: uses IQR. Best when outliers are present.')

## 2. Categorical Encoding

Compare OneHot, Label, Ordinal, and Target encoding on Titanic data.

In [ ]:
# Load Titanic
titanic = sns.load_dataset('titanic').dropna(subset=['survived', 'sex', 'embarked'])
print('Original categorical columns:')
print(titanic[['sex', 'embarked', 'class']].head())

# OneHot Encoding
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded = ohe.fit_transform(titanic[['sex', 'embarked']])
ohe_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
print(f'\nOneHot encoded shape: {ohe_df.shape}')
print(ohe_df.head(3))

In [ ]:
# Target Encoding (manual)
target_mean = titanic.groupby('sex')['survived'].mean()
titanic['sex_target_encoded'] = titanic['sex'].map(target_mean)

print('=== Target Encoding ===')
print('Mean survival by sex:')
print(target_mean)
print('\nSample:')
print(titanic[['sex', 'sex_target_encoded', 'survived']].head(8))

## 3. Missing Value Imputation

Compare imputation strategies on Titanic age data.

In [ ]:
titanic_full = sns.load_dataset('titanic')
print(f'Missing values in Age: {titanic_full["age"].isna().sum()}')

age_data = titanic_full[['age']].copy()

# Mean imputation
mean_imp = SimpleImputer(strategy='mean')
age_mean = mean_imp.fit_transform(age_data)

# Median imputation
median_imp = SimpleImputer(strategy='median')
age_median = median_imp.fit_transform(age_data)

# KNN imputation
knn_imp = KNNImputer(n_neighbors=5)
features_for_knn = titanic_full[['age', 'fare', 'pclass']].copy()
age_knn = knn_imp.fit_transform(features_for_knn)[:, 0]

# Compare distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(age_mean, bins=30, alpha=0.7, color='blue')
axes[0].set_title(f'Mean Imputation (mean={age_mean.mean():.1f})')
axes[1].hist(age_median, bins=30, alpha=0.7, color='green')
axes[1].set_title(f'Median Imputation (median={np.median(age_median):.1f})')
axes[2].hist(age_knn, bins=30, alpha=0.7, color='red')
axes[2].set_title(f'KNN Imputation (mean={age_knn.mean():.1f})')
plt.tight_layout()
plt.show()
print('KNN imputation preserves relationships with other features.')

## 4. Feature Construction

Create interaction, polynomial, binned, and date features.

In [ ]:
# Polynomial and interaction features
X_sample = X_h[['MedInc', 'HouseAge']].iloc[:5]
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X_sample)

poly_df = pd.DataFrame(X_poly, columns=poly.get_feature_names_out())
print('=== Polynomial Features (degree=2) ===')
print(poly_df)

# Binning
kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
age_binned = kbd.fit_transform(titanic_full[['age']].dropna())
print(f'\nBinned age (5 bins, quantile strategy):')
print(pd.Series(age_binned.flatten()).value_counts().sort_index())

In [ ]:
# Date/time feature extraction
dates = pd.date_range('2023-01-01', periods=100, freq='D')
date_df = pd.DataFrame({'date': dates})
date_df['year'] = date_df['date'].dt.year
date_df['month'] = date_df['date'].dt.month
date_df['day'] = date_df['date'].dt.day
date_df['day_of_week'] = date_df['date'].dt.dayofweek
date_df['is_weekend'] = date_df['day_of_week'].isin([5, 6]).astype(int)

print('=== Date/Time Features ===')
print(date_df.head(10))

In [ ]:
# Text features with TF-IDF
documents = [
    'the cat sat on the mat',
    'the dog sat on the log',
    'cats and dogs are pets',
    'the mat was soft and warm'
]

tfidf = TfidfVectorizer(stop_words='english')
X_tfidf = tfidf.fit_transform(documents)

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf.get_feature_names_out())
print('=== TF-IDF Features ===')
print(tfidf_df.round(3))

## 5. Feature Selection

Compare SelectKBest, RFE, and feature_importances_ on Iris.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names

# SelectKBest (ANOVA F-test)
selector = SelectKBest(score_func=f_classif, k=2)
X_kbest = selector.fit_transform(X, y)
kbest_scores = pd.DataFrame({
    'feature': feature_names,
    'f_score': selector.scores_,
    'p_value': selector.pvalues_
}).sort_values('f_score', ascending=False)

print('=== SelectKBest (ANOVA F-test) ===')
print(kbest_scores)
print(f'Selected: {[feature_names[i] for i in selector.get_support(indices=True)]}')

In [ ]:
# RFE (Recursive Feature Elimination)
model = LogisticRegression(max_iter=200, random_state=42)
rfe = RFE(estimator=model, n_features_to_select=2)
rfe.fit(X, y)

rfe_results = pd.DataFrame({
    'feature': feature_names,
    'selected': rfe.support_,
    'rank': rfe.ranking_
}).sort_values('rank')

print('=== RFE Results ===')
print(rfe_results)

# Feature importances from Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print('\n=== Random Forest Feature Importances ===')
print(imp_df)

## 6. Building Feature Pipelines with ColumnTransformer

In [ ]:
# Build a ColumnTransformer pipeline for Titanic
titanic = sns.load_dataset('titanic')
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']].copy()

X = titanic.drop('survived', axis=1)
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define column types
num_cols = ['age', 'fare']
cat_cols = ['pclass', 'sex', 'embarked']

# Build preprocessor
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

# Complete pipeline with classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train and evaluate
pipeline.fit(X_train, y_train)
score = pipeline.score(X_test, y_test)
print(f'=== Pipeline Performance ===')
print(f'Test accuracy: {score:.4f}')
print(f'Number of features after transformation: {pipeline[:-1].transform(X_train).shape[1]}')

## Summary

- **Scaling**: StandardScaler (normal), MinMaxScaler ([0,1]), RobustScaler (outliers)
- **Encoding**: OneHot (nominal), Ordinal (ordinal), Target (high-cardinality, risk of leakage)
- **Imputation**: Mean/median (simple), KNN/Iterative (complex, preserves relationships)
- **Feature creation**: Polynomial, interaction, binning, date/time, text features
- **Feature selection**: Filter (SelectKBest), Wrapper (RFE), Embedded (importances)
- **Pipelines**: ColumnTransformer + Pipeline = reproducible, leak-free feature engineering
- **Kaggle focus**: Feature engineering often matters more than model choice